# VARIANCES

This notebook is dedicated at the empirical study of how the variance of the errors vary among different horizons.

In [1]:
from src.data_handler import *
from src.config_files import *
from src.direct_models import *
from src.mheme import *
from src.metrics import *
from src.plot_handler import *

In [3]:
# This must be setted coherently within the different config files
WINDOW = 48
HORIZON = 24

DATA_PATH = '../data'
DATA_CONFIG_PATH = '../data/data_config.json'

TCN_PATH_CONFIG_LOAD = '../src/config_files/tcn_config.json'
MSE_TCN_PATH_CONFIG_LOAD = '../src/config_files/tcn_config_mse.json'
TCN_PATH_SAVE = '../models/'

XGB_PATH_CONFIG_LOAD = '../src/config_files/xgb_config.json'
XGB_PATH_SAVE = '../models/'

ARIMA_PATH_CONFIG_LOAD = '../src/config_files/arima_config.json'

MODELS_PATH_SAVE = '../models/'

In [16]:
X, data = data_loader(data_path = DATA_PATH, data_config_path= DATA_CONFIG_PATH, dataset_init = 's')
X = data_preprocessing(X, data_config_path=DATA_CONFIG_PATH, dataset_init='s')

C:\Users\Utente\Documents\Università\Multiple Horizons Ensemble Method\src\data_handler.py:190: FutureWarning:

'T' is deprecated and will be removed in a future version, please use 'min' instead.



In [17]:
kwargs = {"name" : "Solar", "color" : "green", "title" : "Solar energy production over time", "x_axis" : "Time", "y_axis" : "Solar energy production"}
plot_time_series(X[-1000:], **kwargs)

In [19]:
X_slide, y_slide = sliding_window(X, window=WINDOW, horizon=HORIZON, k = 7)
train, val, test = train_validation_test_split(X_slide, y_slide, shuffle_data= False, shuffle_internal= True, random_state=42)

In [20]:
mheme_tcn_hahl = UMHEMe(HORIZON, WINDOW, TCN, TCN_PATH_CONFIG_LOAD, skip = 3)

In [21]:
mheme_tcn_hahl.fit(train[0], train[1])

Fitting class : <class 'src.direct_models.TCN'>; horizon : 1


Training TCN: 100%|██████████| 100/100 [00:27<00:00,  3.60it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 4


Training TCN: 100%|██████████| 100/100 [00:27<00:00,  3.67it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 7


...

Training TCN: 100%|██████████| 100/100 [00:30<00:00,  3.31it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 19


Training TCN: 100%|██████████| 100/100 [00:28<00:00,  3.50it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 22


Training TCN: 100%|██████████| 100/100 [00:24<00:00,  4.07it/s]


In [22]:
pre_preds = mheme_tcn_hahl.predict(test[0])
mse(pre_preds, test[1])

tensor(2.5966e+10)

In [23]:
mheme_tcn_hahl.compute_weights(np.concatenate([train[0]], axis = 0), np.concatenate([train[1]], axis = 0))

In [24]:
pre_preds = mheme_tcn_hahl.predict(test[0])
mse(pre_preds, test[1])

tensor(0.2993)

In [26]:
mheme_tcn_hahl.visualize_variances(None)
mheme_tcn_hahl.visualize_errors(None)
mheme_tcn_hahl.visualize_weights(None)

In [ ]:
preds = umheme_tcn_hahl.whole_predict(test[0])
ens_preds = umheme_tcn_hahl.predict(test[0])
for i in range (10):
    pred = {model : preds[model][i] for model in preds}
    plot_multiple_forecast(test[0][i], test[1][i], pred, **kwargs)
    errs = np.array([mse(preds[model][i], test[1][i]) for model in preds])
    avg_mse = np.mean(errs)
    min_mse = np.min(errs)
    max_mse = np.max(errs)
    ensemble_mse = mse(ens_preds[i], test[1][i])
    print(f"min mse : {min_mse}\nmax mse : {max_mse}\naverage mse : {avg_mse}\nensemble mse : {ensemble_mse}")
    

min mse : 12.209582328796387
max mse : 12.951324462890625
average mse : 12.470993995666504
ensemble mse : 12.443425178527832


min mse : 12.134020805358887
max mse : 12.460807800292969
average mse : 12.2617826461792
ensemble mse : 12.230542182922363


min mse : 3.34267520904541
max mse : 3.8932180404663086
average mse : 3.6423397064208984
ensemble mse : 3.45957350730896


min mse : 3.395444869995117
max mse : 3.9191534519195557
average mse : 3.668138265609741
ensemble mse : 3.596677541732788


min mse : 3.267664909362793
max mse : 3.8130569458007812
average mse : 3.5061557292938232
ensemble mse : 3.4655520915985107


min mse : 3.366204261779785
max mse : 3.517580032348633
average mse : 3.4515678882598877
ensemble mse : 3.414926290512085


min mse : 1.6431142091751099
max mse : 2.039930582046509
average mse : 1.8145147562026978
ensemble mse : 1.6467113494873047


min mse : 7.728076934814453
max mse : 9.560673713684082
average mse : 8.686541557312012
ensemble mse : 8.57645320892334


min mse : 8.377470970153809
max mse : 8.936498641967773
average mse : 8.580098152160645
ensemble mse : 8.531025886535645


min mse : 7.938689708709717
max mse : 8.935966491699219
average mse : 8.295509338378906
ensemble mse : 8.23453426361084


In [31]:
preds = umheme_tcn.whole_predict(test[0])
ens_preds = umheme_tcn.predict(test[0])
for i in range (10):
    plot_forecast(test[0][i], test[1][i], ens_preds[i], **kwargs)